# Data cleaning & missing-value handling — corrected pipeline

**Goal of this notebook:** turn `train_V2.csv` and `score.csv` into two clean, model-ready
tables (`train_clean`, `score_clean`) with the *same* columns and transformations.

**Guiding principle:** the three outcome columns are pulled OFF the training set before any
transformation. That way nothing we do to the features can leak or accidentally rescale the
targets, and the final profit predictions stay in euros.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
pd.set_option('display.max_columns', None)

## 1. Load data and separate the outcomes

In [2]:
train = pd.read_csv('train_V2.csv')
score = pd.read_csv('score.csv')

OUTCOME_COLS = ['outcome_profit', 'outcome_damage_inc', 'outcome_damage_amount']

# Keep the targets aside, untouched, indexed so we can re-attach later.
y = train[OUTCOME_COLS].copy()

# Feature matrices only. Tag origin so we can split back after joint cleaning.
train_X = train.drop(columns=OUTCOME_COLS).copy()
train_X['is_train'] = 1
score_X = score.copy()
score_X['is_train'] = 0

datafull = pd.concat([train_X, score_X], axis=0, ignore_index=True)
print('train:', train_X.shape, '| score:', score_X.shape, '| combined:', datafull.shape)

train: (5000, 51) | score: (500, 51) | combined: (5500, 51)


## 2. Survey the missingness

We decide *which columns to drop* using the TRAINING rows only — we never let the applicants
influence a modelling choice.

In [3]:
train_part = datafull[datafull['is_train'] == 1]
miss = (train_part.isnull().mean() * 100).sort_values(ascending=False)
miss[miss > 0].round(1)

score2_pos          75.8
score4_pos          75.5
score1_pos          75.5
score5_pos          75.4
score3_pos          74.8
score2_neg          73.9
score1_neg          73.7
score4_neg          73.5
score3_neg          72.7
score5_neg          70.1
tenure_mts           7.8
tenure_yrs           7.8
neighbor_income      4.8
shop_use             1.8
dining_ic            1.8
cab_requests         1.8
presidential         1.8
profit_am            1.1
income_am            1.1
client_segment       1.1
nights_booked        1.1
claims_no            1.1
company_ic           1.1
bar_no               1.1
sport_ic             1.1
marketing_permit     1.1
age                  1.1
gluten_ic            1.1
credit_use_ic        1.1
lactose_ic           1.1
insurance_ic         1.1
crd_lim_rec          1.1
damage_inc           1.1
profit_last_am       1.1
sect_empl            1.1
gold_status          1.1
urban_ic             1.1
prev_stay            1.1
retired              1.1
fam_adult_size       1.1


### 2a. The staff-score columns are *block*-missing, not randomly missing

All ten `scoreX_pos` / `scoreX_neg` columns are ~70–76% missing. Crucially, the missingness is
structured: a guest tends to be missing *all* their staff scores at once. That almost certainly
means "this guest was never rated by partner hotels" — a meaningful fact, not noise.

So before we drop these columns, we capture the signal as a single feature: how many staff
scores the guest actually has.

In [4]:
score_cols = [c for c in datafull.columns if c.startswith('score')]

# How concentrated is the missingness? (computed on train for reporting)
per_row = train_part[score_cols].isnull().sum(axis=1)
print('Guests missing ALL 10 staff scores:', (per_row == 10).sum())
print('Guests missing NONE:', (per_row == 0).sum())

# Engineered feature: number of staff scores present (0..10). Keeps the information
# that the raw columns are about to lose.
datafull['n_staff_scores'] = datafull[score_cols].notnull().sum(axis=1)

# Now drop the raw score columns (too sparse to impute responsibly).
datafull = datafull.drop(columns=score_cols)
print('Dropped', len(score_cols), 'sparse score columns; kept n_staff_scores instead.')

Guests missing ALL 10 staff scores: 1166
Guests missing NONE: 10
Dropped 10 sparse score columns; kept n_staff_scores instead.


## 3. Remove a redundant feature

`tenure_mts` and `tenure_yrs` are the same thing in different units (correlation ≈ 0.9998,
months ≈ 12 × years). Keeping both adds nothing. We keep months (finer granularity) and drop years.

In [5]:
datafull = datafull.drop(columns=['tenure_yrs'])

## 4. Encode categorical variables

- `gender` (M / V) and `married_cd` (True/False) are binary → 0/1.
- `client_segment` and `sect_empl` are nominal with several levels → one-hot.
  We KEEP all dummy levels (`drop_first=False`) because the models are tree-based
  gradient boosting, which handles redundant dummies fine and reads them more naturally.
- `dummy_na=True` turns "missing segment" into its own informative category.

In [6]:
# Binary
datafull['married_cd'] = datafull['married_cd'].astype(int)
datafull['gender'] = datafull['gender'].map({'M': 0, 'V': 1})  # NaN stays NaN -> imputed later

# Nominal -> one-hot (missing kept as its own column)
for col in ['client_segment', 'sect_empl']:
    datafull[col] = datafull[col].astype('category')
datafull = pd.get_dummies(datafull, columns=['client_segment', 'sect_empl'],
                          drop_first=False, dummy_na=True)

# any bool dummies -> int
bool_cols = datafull.select_dtypes('bool').columns
datafull[bool_cols] = datafull[bool_cols].astype(int)

## 5. Feature engineering

A simple, defensible example: profit per night (normalises a big spender who simply stayed
longer vs. one who is genuinely high-value per night). Log transforms tame the extreme
right-skew of the money variables (skew was ~20–36) so models aren't dominated by a few
outliers. `log1p` is safe at zero.

In [7]:
# Profit per night (guard against divide-by-zero)
datafull['profit_per_night'] = datafull['profit_am'] / datafull['nights_booked'].replace(0, np.nan)

# Log-transform heavily skewed positive money features
skewed_money = ['income_am', 'profit_am', 'profit_last_am', 'damage_am',
                'claims_am', 'shop_am', 'neighbor_income', 'crd_lim_rec']
for col in skewed_money:
    if col in datafull.columns:
        datafull[col + '_log'] = np.log1p(datafull[col].clip(lower=0))

## 6. Outlier handling (required by the exam)

The exam asks us to *identify* outliers and make a deliberate choice. We don't delete rows —
the applicants need a score, and dropping training rows throws away signal. Instead we
**winsorize** (cap) the most extreme numeric values at the 1st/99th percentile, computed on
the training rows only. This makes the analysis robust without losing any guest.

In [8]:
# Numeric continuous columns to cap (exclude binaries/dummies/flags/engineered logs are fine to cap too)
cap_cols = ['income_am', 'profit_am', 'profit_last_am', 'damage_am', 'claims_am',
            'shop_am', 'neighbor_income', 'crd_lim_rec', 'age', 'nights_booked',
            'cab_requests', 'bar_no', 'profit_per_night']
cap_cols = [c for c in cap_cols if c in datafull.columns]

train_mask = datafull['is_train'] == 1
for col in cap_cols:
    lo = datafull.loc[train_mask, col].quantile(0.01)
    hi = datafull.loc[train_mask, col].quantile(0.99)
    datafull[col] = datafull[col].clip(lower=lo, upper=hi)

## 7. Impute remaining missing values

Type-aware: binary columns get the mode, all other numerics get the **median** (robust to the
remaining skew). Stats are taken from the TRAINING rows so the applicants never influence them.

In [9]:
# After encoding, the genuinely binary columns:
binary_vars = ['gender', 'married_cd', 'gold_status', 'marketing_permit', 'urban_ic',
               'prev_stay', 'prev_all_in_stay', 'divorce', 'retired', 'company_ic']
binary_vars = [c for c in binary_vars if c in datafull.columns]

feature_cols = [c for c in datafull.columns if c != 'is_train']

for col in feature_cols:
    if datafull[col].isnull().any():
        if col in binary_vars:
            fill = datafull.loc[train_mask, col].mode()[0]
        else:
            fill = datafull.loc[train_mask, col].median()
        datafull[col] = datafull[col].fillna(fill)

print('Remaining missing values:', int(datafull[feature_cols].isnull().sum().sum()))

Remaining missing values: 0


## 8. Scale the features

We fit the scaler on the training rows and apply it to both, so the applicants are transformed
with exactly the train statistics (no leakage). Note: tree-based gradient boosting does not
*need* scaling, so this step is optional for that model family — but it is harmless and keeps
the pipeline ready for any linear/regularised model you might compare against.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(datafull.loc[train_mask, feature_cols])
datafull[feature_cols] = scaler.transform(datafull[feature_cols])

## 9. Split back and re-attach the untouched outcomes

`is_train` is dropped here so it can never leak into a model.

In [11]:
train_clean = datafull[datafull['is_train'] == 1].drop(columns='is_train').reset_index(drop=True)
score_clean = datafull[datafull['is_train'] == 0].drop(columns='is_train').reset_index(drop=True)

# Re-attach the raw (un-scaled) outcomes to the training set only
train_clean = pd.concat([y.reset_index(drop=True), train_clean], axis=1)

print('train_clean:', train_clean.shape, '| score_clean:', score_clean.shape)
assert list(train_clean.drop(columns=OUTCOME_COLS).columns) == list(score_clean.columns), \
    'Feature columns must match between train and score!'
print('Feature columns match. Outcomes are in euros, untouched.')

train_clean.to_csv('train_clean.csv', index=False)
score_clean.to_csv('score_clean.csv', index=False)

train_clean: (5000, 64) | score_clean: (500, 61)
Feature columns match. Outcomes are in euros, untouched.
